# Feature Engineering

This notebook covers:
1. URL/Path Feature Extraction
2. HTTP Header Analysis
3. Request Body Analysis
4. IP-based Features
5. Time-based Features
6. Encoding & Pattern Detection
7. Feature Scaling & Normalization
8. Feature Selection
9. Final Dataset Preparation

## 1. Import Libraries and Load Data

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import re
import warnings
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

In [16]:
# Load processed data
data_path = Path('../data/processed/v0.1.1/eda_cleaned_data.csv')
df = pd.read_csv(data_path)

print(f"Loaded data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Loaded data shape: (6412, 18)
Columns: ['id', 'timestamp', 'ip', 'method', 'path', 'query', 'body', 'headers', 'prediction', 'label', 'created_at', 'path_length', 'query_length', 'body_length', 'hour', 'day_of_week', 'date', 'header_count']


## 2. URL/Path Feature Extraction

In [17]:
# URL features
df['path_depth'] = df['path'].apply(lambda x: x.count('/') if pd.notna(x) else 0)
df['path_special_chars'] = df['path'].apply(lambda x: len(re.findall(r'[^\w/.-]', x)) if pd.notna(x) else 0)
df['path_encoded_chars'] = df['path'].apply(lambda x: x.count('%') if pd.notna(x) else 0)
df['query_param_count'] = df['query'].apply(lambda x: x.count('&') + (1 if x and x != '' else 0) if pd.notna(x) else 0)
df['query_special_chars'] = df['query'].apply(lambda x: len(re.findall(r'[^\w&=.]', str(x))) if pd.notna(x) else 0)

# Suspicious patterns
suspicious_patterns = ['eval', 'exec', 'system', 'shell', 'cmd', 'bash', 'php', 'sql']
df['suspicious_path'] = df['path'].apply(lambda x: sum(1 for p in suspicious_patterns if p.lower() in x.lower()) if pd.notna(x) else 0)

print("URL Features created")

URL Features created


## 3. HTTP Header Feature Extraction

In [18]:
def parse_headers(header_str):
    if pd.isna(header_str) or header_str == '{}':
        return {}
    try:
        return json.loads(header_str)
    except:
        return {}

df['header_count'] = df['headers'].apply(lambda x: len(parse_headers(x)))
df['user_agent_length'] = df['headers'].apply(lambda x: len(parse_headers(x).get('User-Agent', '')))

def check_suspicious_headers(header_str):
    headers = parse_headers(header_str)
    suspicious_count = 0
    user_agent = headers.get('User-Agent', '').lower()
    if any(tool in user_agent for tool in ['curl', 'wget', 'python', 'sqlmap']):
        suspicious_count += 1
    return suspicious_count

df['suspicious_headers'] = df['headers'].apply(check_suspicious_headers)
print("Header Features created")

Header Features created


## 4. Request Body Analysis

In [23]:
df['body'] = df['body'].fillna('').astype(str)

df['body_contains_code'] = df['body'].apply(
    lambda x: 1 if any(
        kw in x.lower()
        for kw in ['<?php', 'shell_exec', 'base64']
    ) else 0
)

df['body_special_chars_ratio'] = df['body'].apply(
    lambda x: len(re.findall(r'[^\w\s]', x)) / max(len(x), 1)
)

print("Body Features created")

Body Features created


## 5. IP-based Features

In [24]:
ip_attack_count = df[df['label'] == 'Attack'].groupby('ip').size()
ip_total_count = df.groupby('ip').size()

ip_attack_rate = (ip_attack_count / ip_total_count).fillna(0)

df['ip_attack_rate'] = df['ip'].map(ip_attack_rate).fillna(0)
df['ip_request_count'] = df['ip'].map(ip_total_count).fillna(1)
df['is_high_risk_ip'] = (df['ip_attack_rate'] > 0.5).astype(int)

print("IP Features created")

IP Features created


## 6. Time-based Features

In [26]:
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    errors='coerce'  # jika format invalid akan jadi NaT
)

# Isi nilai timestamp yang kosong jika ada
df = df.dropna(subset=['timestamp']).copy()

# Feature waktu
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Jam kerja (09:00 - 17:00)
df['is_business_hours'] = (
    (df['hour'] >= 9) & (df['hour'] <= 17)
).astype(int)

# Jam rawan malam (22:00 - 05:00)
df['is_night_time'] = (
    (df['hour'] >= 22) | (df['hour'] <= 5)
).astype(int)

# Frekuensi request per IP per jam
df['timestamp_hour'] = df['timestamp'].dt.floor('h')

ip_hour_freq = df.groupby(
    ['ip', 'timestamp_hour']
).size()

def get_request_freq(row):
    try:
        return ip_hour_freq.loc[
            (row['ip'], row['timestamp_hour'])
        ]
    except (KeyError, TypeError):
        return 1

df['request_freq_per_hour'] = df.apply(
    get_request_freq,
    axis=1
)

print("Time Features created successfully")

Time Features created successfully


## 7. Encoding & Pattern Detection

In [27]:
df['has_url_encoding'] = df['path'].apply(lambda x: 1 if '%' in x else 0)
df['has_double_encoding'] = df['path'].apply(lambda x: 1 if '%25' in x else 0)
df['has_directory_traversal'] = df['path'].apply(lambda x: 1 if '../' in x else 0)

sql_patterns = ['union', 'select', 'drop', 'insert', 'update', 'delete', 'or', '--']
df['sql_injection_score'] = df['path'].apply(lambda x: sum(1 for p in sql_patterns if p.lower() in x.lower()) if pd.notna(x) else 0)
df['sql_injection_score'] += df['query'].apply(lambda x: sum(1 for p in sql_patterns if p.lower() in str(x).lower()) if pd.notna(x) else 0)

print("Encoding & Pattern Features created")

Encoding & Pattern Features created


## 8. Prepare Feature Set

In [28]:
engineered_features = [
    'path_depth', 'path_special_chars', 'path_encoded_chars', 'query_param_count',
    'query_special_chars', 'suspicious_path', 'header_count', 'suspicious_headers',
    'user_agent_length', 'body_contains_code', 'body_special_chars_ratio',
    'ip_attack_rate', 'ip_request_count', 'is_high_risk_ip',
    'hour', 'day_of_week', 'is_business_hours', 'is_night_time',
    'request_freq_per_hour', 'has_url_encoding', 'has_double_encoding',
    'has_directory_traversal', 'sql_injection_score'
]

print(f"Total engineered features: {len(engineered_features)}")
print(f"\nFeature Statistics:")
print(df[engineered_features].describe())

Total engineered features: 23

Feature Statistics:
        path_depth  path_special_chars  path_encoded_chars  query_param_count  \
count  6412.000000         6412.000000         6412.000000        6412.000000   
mean      4.326887            1.426700            0.223643           0.233001   
std       1.786832            3.922119            1.621745           0.656833   
min       1.000000            0.000000            0.000000           0.000000   
25%       4.000000            0.000000            0.000000           0.000000   
50%       4.000000            0.000000            0.000000           0.000000   
75%       5.000000            0.000000            0.000000           0.000000   
max      17.000000           70.000000           55.000000          13.000000   

       query_special_chars  suspicious_path  header_count  suspicious_headers  \
count          6412.000000      6412.000000   6412.000000         6412.000000   
mean              0.407829         0.752183     26.004835

## 9. Feature Scaling and Selection

In [29]:
X = df[engineered_features].copy().fillna(0)
y = (df['label'] == 'Attack').astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=engineered_features)

# Feature selection
selector = SelectKBest(f_classif, k=min(20, len(engineered_features)))
X_selected = selector.fit_transform(X_scaled, y)

selected_features = [engineered_features[i] for i in selector.get_support(indices=True)]
print(f"Selected {len(selected_features)} features for training")

Selected 20 features for training


## 10. Create Final Dataset

In [30]:
final_dataset = X_scaled_df[selected_features].copy()
final_dataset['label'] = y.values

output_path = Path('../data/processed/v0.1.1/features_engineered_dataset.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
final_dataset.to_csv(output_path, index=False)

print(f"✓ Feature engineered dataset saved: {output_path}")
print(f"Shape: {final_dataset.shape}")
print(f"Attack rate: {final_dataset['label'].mean()*100:.2f}%")

✓ Feature engineered dataset saved: ..\data\processed\features_engineered_dataset.csv
Shape: (6412, 21)
Attack rate: 48.74%
